# 03 — Precipitation Statistics

Storm-relative rainfall summaries around key windows (pre-landfall / landfall / post-landfall)
from the GeoTIFF exported by notebook 01. Requires that export (or a live EE session).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  (registers the .rio accessor)

tif = pathlib.Path('../output/biparjoy_imerg_accum_7-18jun2023.tif')
if not tif.exists():
    raise FileNotFoundError('Run notebook 01 first (needs an authenticated EE session).')
da = rioxarray.open_rasterio(tif, masked=True).squeeze('band', drop=True)
da


In [ ]:
# Mean accumulated rain in 0–100 km and 100–300 km rings around selected track fixes
track = pd.read_csv('../data/biparjoy_besttrack_sample.csv', parse_dates=['time_utc'])

def annular_mean(lon0, lat0, r0_km, r1_km):
    lons, lats = np.meshgrid(da.x.values, da.y.values)
    dx = (lons - lon0) * 111.0 * np.cos(np.radians(lat0))
    dy = (lats - lat0) * 111.0
    r = np.hypot(dx, dy)
    mask = (r >= r0_km) & (r < r1_km) & ~np.isnan(da.values)
    return float(np.mean(da.values[mask])) if mask.any() else np.nan

fixes = {
    'pre-landfall': track.iloc[6],
    'landfall': track.iloc[-2],
    'post-landfall': track.iloc[-1],
}
summary = pd.DataFrame([
    {
        'window': w,
        'core_0_100mm': round(annular_mean(row.lon, row.lat, 0, 100), 1),
        'outer_100_300mm': round(annular_mean(row.lon, row.lat, 100, 300), 1),
    }
    for w, row in fixes.items()
])
summary


**Caveats (keep in any write-up):** IMERG is a satellite retrieval, not gauge truth; coastal pixels
and overpass sampling add bias; track positions carry positional error. Never infer damage or
causality from these maps.
